In [0]:
# ==============================================================================
# SECTION 1: CONFIGURATION & SETUP
# ==============================================================================
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType, IntegerType, StringType

CATALOG_NAME = "diabetes_catalog"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# Root path containing incoming multi-month folders (e.g., 2026_02, 2026_03, etc.)
VOLUME_ROOT = "/Volumes/healthcare_diabetic/hc_schema/hc_volume_incremental"

DATASET_NAMES = [
    "patient",
    "lifestyle",
    "clinical_measurements",
    "diabetes_risk",
    "date",
]

# Create schemas if they do not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}")


# ==============================================================================
# SECTION 2: BRONZE LAYER - AUTO LOADER INGESTION (STREAMING APPEND)
# Reads all incoming CSVs across month subdirectories and appends raw data
# ==============================================================================
def ingest_bronze_autoloader(dataset_name):
    target_table = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.{dataset_name}"
    checkpoint_path = f"{VOLUME_ROOT}/_checkpoints/{dataset_name}"

    print(f"[BRONZE] Ingesting raw streaming data for: {dataset_name}...")

    stream_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("pathGlobFilter", f"*{dataset_name}.csv")
        .load(VOLUME_ROOT)
    )

    processed_df = (
        stream_df.withColumn("source_file_path", F.col("_metadata.file_path"))
        .withColumn(
            "batch_month",
            F.regexp_extract("source_file_path", r"(\d{4}_\d{2})", 1),
        )
        .withColumn("ingested_at", F.current_timestamp())
    )

    query = (
        processed_df.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()
    print(f"[BRONZE] Successfully appended data into {target_table}")


# Execute Bronze Ingestion
for dataset in DATASET_NAMES:
    ingest_bronze_autoloader(dataset)


# ==============================================================================
# SECTION 3: SILVER LAYER - DELTA MERGE UPSERTS (BATCH CLEANSE & DEDUPLICATE)
# Reads appended records from Bronze, cleans, types, and MERGES into Silver
# ==============================================================================
def process_silver_upserts():
    print("\n[SILVER] Starting Silver transformations and MERGE operations...")

    # --- 3A. Silver dim_patient ---
    bronze_patient = (
        spark.read.table(f"{CATALOG_NAME}.{BRONZE_SCHEMA}.patient")
        .select(
            F.col("Patient_ID").cast(DoubleType()).cast(IntegerType()).alias("patient_id"),
            F.col("Age").cast(DoubleType()).cast(IntegerType()).alias("age"),
            F.coalesce(F.col("Gender"), F.lit("Unknown")).alias("gender"),
            F.coalesce(F.col("Country"), F.lit("Unknown")).alias("country"),
            F.coalesce(F.col("Work_Type"), F.lit("Unknown")).alias("work_type"),
            F.coalesce(F.col("Residence_Type"), F.lit("Unknown")).alias(
                "residence_type"
            ),
            F.col("Family_History_Diabetes").alias("family_history_diabetes"),
            F.col("Hypertension").alias("hypertension"),
            F.col("Heart_Disease").alias("heart_disease"),
            F.col("Fatty_Liver").alias("fatty_liver"),
            F.col("PCOS").alias("pcos"),
        )
        .dropDuplicates(["patient_id"])
    )

    if spark.catalog.tableExists(f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_patient"):
        (
            DeltaTable.forName(
                spark, f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_patient"
            )
            .alias("target")
            .merge(
                bronze_patient.alias("source"),
                "target.patient_id = source.patient_id",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        bronze_patient.write.format("delta").saveAsTable(
            f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_patient"
        )

    # --- 3B. Silver dim_lifestyle ---
    bronze_lifestyle = (
        spark.read.table(f"{CATALOG_NAME}.{BRONZE_SCHEMA}.lifestyle")
        .select(
            F.col("Patient_ID").cast(IntegerType()).alias("patient_id"),
            F.col("snapshot_date").cast(DateType()).alias("snapshot_date"),
            F.coalesce(F.col("Physical_Activity_Level"), F.lit("Unknown")).alias(
                "physical_activity_level"
            ),
            F.coalesce(
                F.col("Exercise_Hours_Per_Week").cast(DoubleType()), F.lit(0.0)
            ).alias("exercise_hours_per_week"),
            F.coalesce(
                F.col("Daily_Walking_Minutes").cast(DoubleType()), F.lit(0.0)
            ).alias("daily_walking_minutes"),
            F.coalesce(F.col("Diet_Quality"), F.lit("Unknown")).alias(
                "diet_quality"
            ),
            F.col("Sugar_Intake_Level").alias("sugar_intake_level"),
            F.coalesce(
                F.col("Sleep_Hours").cast(DoubleType()), F.lit(7.0)
            ).alias("sleep_hours"),
            F.col("Stress_Level").alias("stress_level"),
            F.col("Smoking_Status").alias("smoking_status"),
            F.col("Alcohol_Consumption").alias("alcohol_consumption"),
            F.coalesce(F.col("Medication_Adherence"), F.lit("Unknown")).alias(
                "medication_adherence"
            ),
            F.col("Daily_Water_Intake_L")
            .cast(DoubleType())
            .alias("daily_water_intake_l"),
        )
        .dropDuplicates(["patient_id", "snapshot_date"])
    )

    if spark.catalog.tableExists(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_lifestyle"
    ):
        (
            DeltaTable.forName(
                spark, f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_lifestyle"
            )
            .alias("target")
            .merge(
                bronze_lifestyle.alias("source"),
                "target.patient_id = source.patient_id AND target.snapshot_date = source.snapshot_date",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        bronze_lifestyle.write.format("delta").saveAsTable(
            f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_lifestyle"
        )

    # --- 3C. Silver fact_clinical_measurements ---
    bronze_clinical = (
        spark.read.table(
            f"{CATALOG_NAME}.{BRONZE_SCHEMA}.clinical_measurements"
        )
        .select(
            F.col("Patient_ID").cast(IntegerType()).alias("patient_id"),
            F.col("snapshot_date").cast(DateType()).alias("snapshot_date"),
            F.col("Height_cm").cast(DoubleType()).alias("height_cm"),
            F.col("Weight_kg").cast(DoubleType()).alias("weight_kg"),
            F.col("BMI").cast(DoubleType()).alias("bmi"),
            F.col("Waist_Circumference_cm")
            .cast(DoubleType())
            .alias("waist_circumference_cm"),
            F.col("Blood_Glucose").cast(DoubleType()).alias("blood_glucose"),
            F.col("HbA1c").cast(DoubleType()).alias("hba1c"),
            F.col("Fasting_Blood_Sugar")
            .cast(DoubleType())
            .alias("fasting_blood_sugar"),
            F.col("Insulin_Level").cast(DoubleType()).alias("insulin_level"),
            F.col("Blood_Pressure_Systolic")
            .cast(IntegerType())
            .alias("bp_systolic"),
            F.col("Blood_Pressure_Diastolic")
            .cast(IntegerType())
            .alias("bp_diastolic"),
            F.col("Total_Cholesterol")
            .cast(DoubleType())
            .alias("total_cholesterol"),
            F.col("HDL").cast(DoubleType()).alias("hdl"),
            F.col("LDL").cast(DoubleType()).alias("ldl"),
            F.col("Triglycerides").cast(DoubleType()).alias("triglycerides"),
            F.col("Heart_Rate").cast(IntegerType()).alias("heart_rate"),
        )
        # Apply mathematical imputation formula for missing height/weight
        .withColumn(
            "height_cm",
            F.when(
                F.col("height_cm").isNull()
                & F.col("weight_kg").isNotNull()
                & F.col("bmi").isNotNull(),
                F.sqrt(F.col("weight_kg") / F.col("bmi")) * 100,
            ).otherwise(F.col("height_cm")),
        )
        .withColumn(
            "weight_kg",
            F.when(
                F.col("weight_kg").isNull()
                & F.col("height_cm").isNotNull()
                & F.col("bmi").isNotNull(),
                F.col("bmi") * F.pow(F.col("height_cm") / 100, 2),
            ).otherwise(F.col("weight_kg")),
        )
        .dropDuplicates(["patient_id", "snapshot_date"])
    )

    if spark.catalog.tableExists(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_clinical_measurements"
    ):
        (
            DeltaTable.forName(
                spark,
                f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_clinical_measurements",
            )
            .alias("target")
            .merge(
                bronze_clinical.alias("source"),
                "target.patient_id = source.patient_id AND target.snapshot_date = source.snapshot_date",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        bronze_clinical.write.format("delta").saveAsTable(
            f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_clinical_measurements"
        )

    # --- 3D. Silver fact_diabetes_risk ---
    bronze_risk = (
        spark.read.table(f"{CATALOG_NAME}.{BRONZE_SCHEMA}.diabetes_risk")
        .select(
            F.col("Patient_ID").cast(IntegerType()).alias("patient_id"),
            F.col("snapshot_date").cast(DateType()).alias("snapshot_date"),
            F.col("Diabetes_Risk_Score")
            .cast(IntegerType())
            .alias("diabetes_risk_score"),
            F.col("AI_Health_Recommendation").alias(
                "ai_health_recommendation"
            ),
            F.col("Doctor_Consultation_Needed").alias(
                "doctor_consultation_needed"
            ),
            F.col("Diabetes_Risk").alias("diabetes_risk"),
        )
        .dropDuplicates(["patient_id", "snapshot_date"])
    )

    if spark.catalog.tableExists(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_diabetes_risk"
    ):
        (
            DeltaTable.forName(
                spark, f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_diabetes_risk"
            )
            .alias("target")
            .merge(
                bronze_risk.alias("source"),
                "target.patient_id = source.patient_id AND target.snapshot_date = source.snapshot_date",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        bronze_risk.write.format("delta").saveAsTable(
            f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_diabetes_risk"
        )

    print("[SILVER] All Silver tables updated and deduplicated successfully.")


# Execute Silver Processing
process_silver_upserts()


# ==============================================================================
# SECTION 4: GOLD LAYER - BUSINESS AGGREGATIONS & TIME-SERIES RECALCULATION
# Re-aggregates full Silver dataset across all months (Jan to June)
# ==============================================================================
def process_gold_aggregations():
    print("\n[GOLD] Recalculating Gold Layer Analytics...")

    dim_patient = spark.read.table(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_patient"
    )
    dim_lifestyle = spark.read.table(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.dim_lifestyle"
    )
    fact_clinical = spark.read.table(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_clinical_measurements"
    )
    fact_risk = spark.read.table(
        f"{CATALOG_NAME}.{SILVER_SCHEMA}.fact_diabetes_risk"
    )

    # 4A. Patient Health 360 View
    gold_patient_360 = (
        fact_clinical.alias("fc")
        .join(dim_patient.alias("dp"), on="patient_id", how="inner")
        .join(
            fact_risk.alias("fr"),
            on=["patient_id", "snapshot_date"],
            how="inner",
        )
        .join(
            dim_lifestyle.alias("dl"),
            on=["patient_id", "snapshot_date"],
            how="inner",
        )
        .select(
            F.col("patient_id"),
            F.col("snapshot_date"),
            F.col("dp.age"),
            F.col("dp.gender"),
            F.col("dp.country"),
            F.col("fc.bmi"),
            F.when(F.col("fc.bmi") < 18.5, "Underweight")
            .when(
                (F.col("fc.bmi") >= 18.5) & (F.col("fc.bmi") < 25.0), "Normal"
            )
            .when(
                (F.col("fc.bmi") >= 25.0) & (F.col("fc.bmi") < 30.0),
                "Overweight",
            )
            .otherwise("Obese")
            .alias("bmi_category"),
            F.col("fc.hba1c"),
            F.when(F.col("fc.hba1c") < 5.7, "Normal")
            .when(
                (F.col("fc.hba1c") >= 5.7) & (F.col("fc.hba1c") <= 6.4),
                "Prediabetes",
            )
            .otherwise("Diabetes")
            .alias("hba1c_condition"),
            F.col("dl.physical_activity_level"),
            F.col("dl.diet_quality"),
            F.col("dl.smoking_status"),
            F.col("fr.diabetes_risk_score"),
            F.col("fr.diabetes_risk"),
            F.col("fr.doctor_consultation_needed"),
        )
    )

    gold_patient_360.write.format("delta").mode("overwrite").saveAsTable(
        f"{CATALOG_NAME}.{GOLD_SCHEMA}.patient_health_360"
    )

    # 4B. Lifestyle Risk Matrix
    gold_lifestyle_matrix = (
        gold_patient_360.groupBy(
            "physical_activity_level", "diet_quality", "smoking_status"
        )
        .agg(
            F.count("patient_id").alias("patient_cohort_size"),
            F.round(F.avg("bmi"), 2).alias("avg_bmi"),
            F.round(F.avg("diabetes_risk_score"), 2).alias("avg_risk_score"),
            F.sum(
                F.when(F.col("hba1c_condition") == "Diabetes", 1).otherwise(0)
            ).alias("diabetic_cases_count"),
        )
        .withColumn(
            "diabetes_prevalence_rate",
            F.round(
                (F.col("diabetic_cases_count") / F.col("patient_cohort_size"))
                * 100,
                2,
            ),
        )
    )

    gold_lifestyle_matrix.write.format("delta").mode("overwrite").saveAsTable(
        f"{CATALOG_NAME}.{GOLD_SCHEMA}.lifestyle_risk_matrix"
    )

    print(
        "[GOLD] Successfully refreshed patient_health_360 and lifestyle_risk_matrix tables."
    )


# Execute Gold Processing
process_gold_aggregations()